# 实验 13 — 测试 dlt pipeline

**目标**：学会给 dlt pipeline 写单元/集成测试，让它进入 CI。

dlt 没有专门的“测试框架”，但提供了几个钩子：

- `dev_mode=True` ：每次跑都生成带时间戳的隔离 dataset，跑完不污染主数据
- `pipeline.run()` 返回 `LoadInfo`，可以断言：行数、状态、是否有失败 jobs
- `pipeline.drop()` ：清掉一个 pipeline 的所有 state 和 schema，便于重跑
- 用 in-memory DuckDB（`:memory:`）做单元测试，秒级

下面在 notebook 里直接演示，搬到 pytest 里几乎是一对一对应。

In [ ]:
# 1) 单元测试：拿 in-memory DuckDB 跑一个最小 pipeline
import dlt
from typing import Iterator

@dlt.resource(write_disposition='merge', primary_key='id', name='items')
def items() -> Iterator[dict]:
    yield from [{'id':1,'name':'a'}, {'id':2,'name':'b'}, {'id':1,'name':'a2'}]  # 第3行更新第1行

pipe = dlt.pipeline(
    pipeline_name='test_pipeline',
    destination=dlt.destinations.duckdb(':memory:'),
    dataset_name='test',
    dev_mode=True,  # 每次跑生成新 dataset_name__YYYYMMDDHHMMSS
)

info = pipe.run(items())
print('has_failed_jobs:', info.has_failed_jobs)
print('loads_ids:', info.loads_ids)

In [ ]:
# 2) 断言行数：用 pipeline.sql_client() 直接查
with pipe.sql_client() as cli:
    rows = list(cli.execute_sql('select id, name from items order by id'))
    print('rows:', rows)

assert len(rows) == 2, '应该 2 行（id=1 被 merge 更新）'
assert dict(zip(['id','name'], rows[0])) == {'id':1, 'name':'a2'}, 'merge 应该用最新的值'
print('OK')

In [ ]:
# 3) 测试异常路径：让 resource 抛错，断言 LoadInfo.has_failed_jobs
import dlt
@dlt.resource(name='items')
def bad_items():
    yield {'id':1}
    raise RuntimeError('upstream API failed')

pipe2 = dlt.pipeline('bad', destination=dlt.destinations.duckdb(':memory:'),
                    dataset_name='t', dev_mode=True)
try:
    info = pipe2.run(bad_items())
    print('finished without exception, failed?', info.has_failed_jobs)
except Exception as e:
    print('extract raised:', type(e).__name__, str(e)[:200])

# 关键点：dlt 把 extract / normalize / load 分阶段处理，extract 阶段抛错会向外冒泡
# load 阶段失败（destination 写不进去）会被 dlt 收进 LoadInfo.failed_jobs 并向外抛 PipelineStepFailed

In [ ]:
# 4) pipeline.drop() 重置 state，常用于测试 setup / teardown
pipe3 = dlt.pipeline('to_drop', destination=dlt.destinations.duckdb(':memory:'),
                    dataset_name='t')
pipe3.run(items())
print('before drop, datasets:', pipe3.dataset().row_counts() if hasattr(pipe3, 'dataset') else '...')
pipe3.drop()
print('after drop OK')

## 生产环境踩坑提示

- **pytest 模板**：
  ```python
  def test_items_merge(tmp_path):
      pipe = dlt.pipeline('test', destination=dlt.destinations.duckdb(':memory:'),
                          dataset_name='t', dev_mode=True)
      info = pipe.run(items())
      assert not info.has_failed_jobs
      with pipe.sql_client() as cli:
          assert cli.execute_sql('select count(*) from items')[0][0] == 2
  ```
- **VCR / responses**：源是 REST API，要用 `vcrpy` 或 `responses` 拦截 HTTP，避免 CI 依赖外网。
- **dev_mode** 会在 dataset_name 后加时间戳。生产 pipeline 千万不要打开 dev_mode，会每跑一次建一个新 dataset。
- **`pipeline.dataset()` API**：dlt 1.5+ 提供了 `pipe.dataset()` 直接拿到 Ibis-like 查询接口，可以直接 `.row_counts()`、`.table('items').head()`，比每次 sql_client 顺手。
- **Schema diff 测试**：把 `pipe.default_schema.to_pretty_yaml()` 在测试里和一个固定 fixture 比对，schema drift 会立刻被发现。

## 思考题

- 给 `ecb_rates.py` 写一个 pytest：mock Frankfurter API（用 `responses`），断言写入 3 个货币 × 2 天 = 6 行。
- `dev_mode=True` 真的能让两次测试不互相干扰吗？读读时间戳精度——如果两次测试相隔 < 1 秒会怎样？
- 如何在 CI 里检测 schema drift，使得任何 raw 表新增列都需要人审批？